# 🚲 자전거 대여량 예측 (GRU) 시계열 오류 분석 Q&A 보고서

본 노트는 **[잘못된 사례]**와 **[보완된 사례]**의 핵심 코드 및 Q&A 보고서 내용을 직관적인 주석과 햄버거 비유로 정리한 과제 제출용 파일입니다.

---

## 📌 [문제 1] 학습 데이터 분리 단계 (Data Splitting)

### ❌ 잘못된 사례 코드
```python
from sklearn.model_selection import train_test_split

# X, y는 시계열 순서대로 정렬된 상태라고 가정
# ⚠️ 오류: 시계열 데이터에 무작위(Random) 셔플 분리 적용
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
```

### 💡 Q1. 발생 원인 및 문제점 (Cause & Effect)
* **문제점 (Cause):** `train_test_split`은 데이터를 **무작위(Random Shuffle)**로 섞어서 나누기 때문에 미래의 데이터가 학습용(`train`)으로 들어가고 과거의 데이터가 검증용(`val`)으로 섞이게 됩니다.
* **부작용 (Effect):** 미래의 정보가 과거 학습 시점에 미리 흘러들어가는 **데이터 누수(Data Leakage / Look-ahead bias)**가 발생합니다. 이로 인해 검증 점수는 말도 안 되게 좋게 나오지만, 실제 미래 예측 시에는 모델이 멘붕을 겪으며 점수가 떡락합니다.

### ✅ 보완된 사례 (해결 코드)
```python
# 🍔 햄버거 장부 비유: 시간 순서대로 장부 자르기 (앞 80%는 과거 학습용, 뒤 20%는 미래 검증용)
split_idx = int(len(X) * 0.8)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]
```

---

## 📌 [문제 2] GRU 입력 데이터 형태 변환 (Reshape / Dimensions)

### ❌ 잘못된 사례 코드
```python
import numpy as np

# X의 shape: (Rows, 8) -> 8개 컬럼(특성)
# ⚠️ 오류: 3차원으로 맞춘답시고 특성(8개)을 Time Step(시간 축)으로 엉뚱하게 넘김
X_train_wrong = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
# model.add(GRU(64, input_shape=(8, 1)))
```

### 💡 Q2. 모델의 오해 및 데이터 해석 오류
* **문제점 (Cause):** 3차원 입력 구조 `(샘플 수, time_steps, features)`에서 `input_shape=(8, 1)`로 설정하면, 8개의 서로 다른 변수(온도, 습도, 풍속 등)를 8번의 **'시간(Time Step)'** 흐름으로 착각하게 됩니다.
* **부작용 (Effect):** 모델은 "온도 → 습도 → 풍속" 순서로 시간이 흐른다고 잘못 학습하여 변수 간의 물리적 의미와 시간적 연관성을 완전히 엉뚱하게 해석합니다.

### ✅ 보완된 사례 (해결 코드)
```python
# 🍔 햄버거 비유: 과거 24시간 간격(시퀀스)을 시간 축으로 삼고, 8개 재료(특성)를 한 타임스텝에 한 묶음으로 넣기
# 올바른 3차원 구조: (샘플 수, 시간 길이=24, 특성 수=8)
# input_shape = (24, 8)
# model.add(GRU(64, input_shape=(24, 8)))
```

---

## 📌 [문제 3] 테스트 데이터 예측 시점 (Test Sequence Boundary)

### ❌ 잘못된 사례 코드
```python
# ⚠️ 오류: test.csv 파일 단독으로 24시간 시퀀스를 바로 생성
# X_test_seq = create_sequences(X_test, time_steps=24)
```

### 💡 Q3. 문제 이유 및 해결 방법
* **문제점 (Cause):** 20일 00시(테스트 데이터의 맨 첫 번째 시간)를 예측하려면 **그 직전 24시간(19일 00시~23시)**의 과거 데이터가 필요합니다. 하지만 `test.csv`만 단독으로 잘라 쓰면 맨 앞 23개 행은 과거 데이터 부족으로 시퀀스를 못 만들어 버려지거나 예측 불가 상태가 됩니다.
* **해결 방법 (Solution):** 학습 데이터(train)의 **가장 마지막 23개 행을 `test.csv` 맨 앞에 붙인 뒤** 시퀀스를 생성해야 테스트 데이터의 첫 번째 시간부터 손실 없이 완벽하게 예측할 수 있습니다.

### ✅ 보완된 사례 (해결 코드)
```python
# 🍔 햄버거 비유: 어제 장부 마지막 23시간 치를 오늘 장부 맨 앞에 붙여서 끊김 없이 연결하기
# train_data의 끝부분(time_steps - 1)과 test_data를 합친 후 시퀀스 생성
# combined_test = pd.concat([train_data.tail(23), test_data], axis=0)
# X_test_seq = create_sequences(combined_test, time_steps=24)
```

---

## 📌 [문제 4] 예측 결과의 도메인 적합성 (Post-processing)

### ❌ 잘못된 사례 코드
```python
# ⚠️ 오류: 모델이 내뱉은 예측값을 음수 검사 없이 그대로 파일 제출
# submission['count'] = y_pred
# submission.to_csv('result.csv')
```

### 💡 Q4. 제출 불가 이유 및 후처리 코드
* **문제점 (Cause):** GRU 회귀(Regression) 모델은 수식 계산 특성상 0 이하의 음수(`-5`, `-12` 등)를 출력할 수 있습니다. 자전거 대여량은 현실에서 **절대 음수가 될 수 없는 물리적 수치(0 이상)**입니다.
* **부작용 (Effect):** 음수가 포함된 채 제출하면 도메인 규칙 위반으로 **평가 지표(RMSLE 등)에서 에러(NaN/Inf)가 발생**하거나 점수가 통째로 감점됩니다.

### ✅ 보완된 사례 (해결 코드)
```python
import numpy as np

# 🍔 햄버거 비유: 0대 미만으로 내려간 건 말 안 되니까 최소 0대로 딱 잘라 맞추기 (후처리)
# np.maximum(0, y_pred) 또는 np.clip(y_pred, 0, None) 사용
y_pred_clean = np.maximum(0, y_pred)

# submission['count'] = y_pred_clean
# submission.to_csv('result.csv')
```
